# Week 6 · The supply chain lab: the wave and the mill question

**Sentence of the day:** *A supply chain is a line of firms that only see the order in front of them. The fewer the hands and the better the news that travels back, the smaller the wave.*

**What you will do today (about 60 minutes)**

| Part | What | Time | You type |
|---|---|---|---|
| 0 | Setup | 2 min | nothing |
| 1 | ▶ The wave: this morning's game as a computer model, with two sliders | 15 min | nothing – you play |
| 2 | ✏️ The mill question: four charts for Aylin | 30 min | five short blanks |
| 3 | 🧩 Your one-page dashboard | 10 min | nothing |
| 4 | ⭐ Extra: 1,000 years of flour risk | if time | nothing |

**How this notebook works**

- Google Colab is a notebook: a document where text and runnable code sit side by side. You run a code cell with the ▶ button on its left, or with **Shift + Enter**.
- Run the cells **from top to bottom**. A cell can only use what the cells above it have created.
- Lines that start with `#` are **comments**: notes in plain English for you. Python skips them.
- A cell marked **✏️ YOUR TURN** has one blank `___`. Replace it, run the cell, then run the **CHECK** cell below it: ✅ = right, ❌ = look at the hint.
- A **slider** is a little bar you drag with the mouse. The chart redraws each time you let go.

## Part 0 · Setup

This cell loads the **libraries** (toolboxes other people have written): `pandas` for tables, `numpy` for numbers, `matplotlib` for charts, `ipywidgets` for sliders. All are installed in Colab. It also defines `check`, the helper that prints ✅ or ❌. Run it once.

In [ ]:
# ── Part 0 · Setup ──────────────────────────────────────────────────────────────
# A line that starts with # is a COMMENT: a note for humans. Python skips it.
import math                       # basic maths
import pandas as pd               # tables with named columns ("DataFrames"), like a spreadsheet
import numpy as np                # lists of numbers and random numbers
import matplotlib.pyplot as plt   # drawing charts
from ipywidgets import interact, FloatSlider, IntSlider, Checkbox   # sliders and tick boxes

# Our colours (hex codes: #rrggbb). Names in CAPITALS = set once, never changed.
GREY, RED, DARK, GREEN, LIGHT = "#a6a6a6", "#c1002a", "#1a1a1a", "#2e7d4f", "#f2f2f2"
STAGE_COLOURS = ["#c9c9c9", "#9a9a9a", "#5f5f5f", "#1a1a1a"]      # Store light grey … Farm black
MILL_COLOURS = {"Bergheim": GREY, "Hofmühle": GREEN, "Großmühle": DARK}

# One chart style for the whole notebook.
plt.rcParams.update({"figure.facecolor": "white", "axes.facecolor": "white", "font.size": 11,
                     "axes.spines.top": False, "axes.spines.right": False})

# (Technical – you may skip this.) Make an unfilled ___ stop the cell with a clear message.
try:
    get_ipython().displayhook.cache_size = 0
    get_ipython().user_ns.pop("___", None)
except NameError:
    pass


def check(label, got, expected, tol=0.01, hint=""):
    """Compare your result with the hand calculation and print ✅ or ❌."""
    try:
        ok = abs(float(got) - float(expected)) <= tol      # True when the two are close enough
        shown = f"{float(got):,.2f}"
    except Exception:
        ok, shown = False, str(got)
    print(("✅ " if ok else "❌ ") + f"{label}: you have {shown}, expected {expected:,.2f}"
          + ("" if ok else f"   Hint: {hint}"))


def not_yet(error):
    # Turns Python's error message into a plain hint – usually: a ___ blank is still waiting.
    if "___" in str(error) or isinstance(error, NameError):
        print("❌ Not done yet: the cell above has not run to its end. Replace the ___ there, run it, then run this check again.")
    else:
        print("❌ Not working yet:", error)


print("Setup done.")

## Part 1 · ▶ The wave: this morning's game as a computer model

This morning you played the **Bread chain game**: Store → Bakery → Mill → Farm, orders travel up, goods travel down, 1 round order delay + 2 rounds shipping delay, customers order **4 cases a round, then 8 from round 5**. Cost = 0.50 € per case in stock + 1.00 € per case still owed (backlog).

The cell below is the **same game as a model**: four computer players who each use the same simple ordering rule that Sterman (1989) measured in real players:

**order = my guess of the next order + how nervous × (the stock I want − the stock I have − a part of what is already on its way)**

- **How nervous** = how hard a player corrects a gap in stock. 0 = never corrects, 1 = corrects the whole gap at once. Real players: about 0.26.
- **Share information** = every stage sees what customers buy at the store, and counts *everything* already on its way.

You do not need to read the code. Run it, then **play with the two controls** in the next cell.

In [ ]:
# ── 1a · The chain as a model (the same rules as the browser game) ────────────────
def rnd(x):
    return math.floor(x + 0.5)          # round x.5 up, as the browser game does

def beer_game(nervous=0.26, shared=False, rounds=24):
    """Play 24 rounds with four computer players. Returns the orders, stock and cost."""
    stock = [12] * 4                     # cases in stock at Store, Bakery, Mill, Farm
    backlog = [0] * 4                    # cases each stage still owes its customer
    pipe = [[4, 4] for _ in range(4)]    # goods on the truck: arrive in 1 round, in 2 rounds
    order_in = [4] * 4                   # the order that reaches each stage this round
    on_order = [12] * 4                  # ordered but not yet arrived
    guess = [4.0] * 4                    # each player's guess of the next order
    field = 4                            # the Farm's order to the field
    H = {"demand": [], "orders": [[] for _ in range(4)], "net": [[] for _ in range(4)], "cost": [0.0] * 4}
    for t in range(rounds):
        D = 4 if t < 4 else 8                                   # what customers buy at the store
        for i in range(4):                                      # 1) goods arrive
            arrived = pipe[i].pop(0); stock[i] += arrived; on_order[i] -= arrived
        shipped, incoming = [0] * 4, [0] * 4
        for i in range(4):                                      # 2) orders arrive, goods are shipped
            incoming[i] = D if i == 0 else order_in[i]
            need = incoming[i] + backlog[i]
            shipped[i] = min(stock[i], need)
            stock[i] -= shipped[i]; backlog[i] = need - shipped[i]
        for i in range(3):
            pipe[i].append(shipped[i + 1])                      # what the supplier ships goes on the truck
        pipe[3].append(field)
        for i in range(4):                                      # 3) cost of this round
            H["cost"][i] += 0.5 * stock[i] + 1.0 * backlog[i]
            H["net"][i].append(stock[i] - backlog[i])
        H["demand"].append(D)
        orders = []
        for i in range(4):                                      # 4) every stage places its order
            if shared:   # sees customer demand, counts all that is on its way
                o = D + nervous * (12 - (stock[i] - backlog[i])) + nervous * (2 * D - on_order[i])
            else:        # sees only the order in front of it (Sterman's rule)
                guess[i] = 0.36 * incoming[i] + 0.64 * guess[i]
                o = guess[i] + nervous * (15 - (stock[i] - backlog[i]) - 0.34 * on_order[i])
            o = max(0, rnd(o)); orders.append(o); on_order[i] += o; H["orders"][i].append(o)
        for i in range(3):
            order_in[i + 1] = orders[i]
        field = orders[3]
    return H

STAGES = ["Store", "Bakery", "Mill", "Farm"]
H = beer_game()
print("Whole-chain cost with four computer players:", f"{sum(H['cost']):,.2f} €")
print("Smallest and biggest order per stage:", {s: (min(o), max(o)) for s, o in zip(STAGES, H["orders"])})

In [ ]:
# CHECK 1a – the model must give the same result as the browser game (normal rules)
try:
    check("chain cost, normal rules", sum(H["cost"]), 1021.50, hint="Run the cell above again.")
    check("Farm's biggest order", max(H["orders"][3]), 26, tol=0)
except Exception as e:
    not_yet(e)

### 1b · ▶ Play with the wave

Run the cell. Then move the slider and tick the box. Watch two things: **the lines on the left** (does the Farm's line swing much more than the customers' green line?) and **the bars on the right** (the range = biggest − smallest order; a growing staircase = the **bullwhip effect**).

In [ ]:
# ── 1b · Two controls, one picture ──────────────────────────────────────────────
def show_wave(how_nervous=0.26, share_information=False):
    H = beer_game(nervous=how_nervous, shared=share_information)
    fig, (left, right) = plt.subplots(1, 2, figsize=(13, 4.5), gridspec_kw={"width_ratios": [3, 2]})
    rounds = range(1, 25)
    left.plot(rounds, H["demand"], color=GREEN, lw=3, ls="--", label="Customers")
    for i, s in enumerate(STAGES):
        left.plot(rounds, H["orders"][i], color=STAGE_COLOURS[i], lw=2.5, label=s)
    left.set(title="Orders placed by each stage, round by round", xlabel="round", ylabel="cases ordered")
    left.legend(frameon=False, ncol=5, loc="upper left")
    ranges = [4] + [max(o) - min(o) for o in H["orders"]]          # customers' range is 8 − 4 = 4
    bars = right.bar(["Customers"] + STAGES, ranges, color=[GREEN] + STAGE_COLOURS)
    right.bar_label(bars, fontsize=12, fontweight="bold")
    right.set(title="The wave: range of orders (biggest − smallest)", ylabel="cases")
    right.set_ylim(0, 30)
    fig.suptitle(f"Whole-chain cost: {sum(H['cost']):,.2f} €   ·   "
                 + ("information shared" if share_information else "each stage sees only its own orders"),
                 fontsize=14, fontweight="bold", color=RED)
    plt.tight_layout(); plt.show()

interact(show_wave,
         how_nervous=FloatSlider(value=0.26, min=0.05, max=1.0, step=0.05, description="how nervous", continuous_update=False),
         share_information=Checkbox(value=False, description="share information"));

**Talk in your team (2 minutes), then write one line each on your sheet:**

1. With *share information* off: which stage has the biggest range? How many times the customers' range?
2. Move *how nervous* to 0.10 and to 0.80. Which one makes the wave bigger – and why does correcting fast make it worse?
3. Tick *share information*. What happens to the staircase and to the cost? Which cure from the chapter is this (point-of-sale data, VMI, CPFR)?

## Part 2 · ✏️ The mill question: four charts for Aylin

Bergheim has just called: flood, three weeks late. Aylin must sign a flour contract for next year by Monday. Kaya needs **39.6 tonnes of flour a year** (3,300 rolls a night × 40 g × 300 nights). Here is Lena's table of the three offers (Exhibit A of the case).

In [ ]:
# ── 2 · The three offers as a table ────────────────────────────────────────────
NEED_T = 3300 * 40 * 300 / 1_000_000      # tonnes of flour a year (g → t: divide by 1,000,000)

mills = pd.DataFrame({
    "km":        [150, 30, 320],          # distance from the mill to Kaya
    "price":     [560, 580, 490],         # € per tonne at the mill
    "freight":   [25, 10, 75],            # € per tonne for the truck
    "days":      [3, 1, 10],              # days the flour is on its way (Kaya already owns it)
    "on_time %": [92, 98, 95],
    "lot t":     [1.0, 0.5, 10.0],        # tonnes per delivery
    "max share": ["100 %", "60 %", "100 %"],
}, index=["Bergheim", "Hofmühle", "Großmühle"])

print(f"Flour need: {NEED_T} t a year")
mills

### 2a · Factor rating: the best supplier on five criteria

**Weighted score = Σ (weight × score).** Weights add up to 1.00, scores go from 1 (bad) to 10 (good). The highest weighted score wins.

The colour table shows Lena's scores: dark = good. **✏️ YOUR TURN:** multiply the scores by the weights and add up each row. Replace the `___` with the name of the weights.

In [ ]:
# ── 2a · Lena's scores and the chapter's weights ─────────────────────────────────
weights = pd.Series({"price": 0.30, "reliability": 0.25, "lead time": 0.15, "CO₂": 0.15, "lot size": 0.15})
scores = pd.DataFrame({
    "price":       [6, 5, 10],
    "reliability": [6, 9, 7],
    "lead time":   [7, 10, 3],
    "CO₂":         [6, 10, 3],
    "lot size":    [7, 9, 2],
}, index=mills.index)

# Picture 1: the scores as a colour table
fig, ax = plt.subplots(figsize=(7, 2.6))
ax.imshow(scores.values, cmap="Greens", vmin=0, vmax=10)
ax.set_xticks(range(5), [f"{c}\n(weight {w:.2f})" for c, w in weights.items()])
ax.set_yticks(range(3), scores.index)
for r in range(3):
    for c in range(5):
        ax.text(c, r, scores.values[r, c], ha="center", va="center", fontsize=13, fontweight="bold",
                color="white" if scores.values[r, c] >= 7 else DARK)
ax.set_title("Lena's scores (1 = bad, 10 = good)"); plt.tight_layout(); plt.show()

# ✏️ YOUR TURN: score × weight, column by column, then add up each row (sum(axis=1) = across)
weighted = (scores * ___).sum(axis=1)
print(weighted.round(2))

In [ ]:
# CHECK 2a
try:
    check("Bergheim", weighted["Bergheim"], 6.3, hint="weighted = (scores * weights).sum(axis=1)")
    check("Hofmühle", weighted["Hofmühle"], 8.1)
    check("Großmühle", weighted["Großmühle"], 5.95)
except Exception as e:
    not_yet(e)

**▶ Play:** each bar below is built from five pieces (weight × score). Drag the **price weight** up: at what weight does the cheap Großmühle win? (The other four weights shrink so all five still add up to 1.00.)

In [ ]:
# ── 2a · Where do the points come from? ─────────────────────────────────────────
def show_rating(price_weight=0.30):
    w = weights.copy()
    rest = w.drop("price")
    w[rest.index] = rest / rest.sum() * (1 - price_weight)     # share the rest in the old proportions
    w["price"] = price_weight
    parts = scores * w                                          # one piece per criterion
    fig, ax = plt.subplots(figsize=(9, 3.2))
    left = np.zeros(3)
    shades = ["#c1002a", "#e07a8c", "#9a9a9a", "#2e7d4f", "#1a1a1a"]
    for (crit, col), shade in zip(parts.items(), shades):
        ax.barh(parts.index, col, left=left, color=shade, label=crit)
        left += col.values
    for y, total in enumerate(left):
        ax.text(total + 0.08, y, f"{total:.2f}", va="center", fontsize=13, fontweight="bold")
    ax.set_xlim(0, 10); ax.invert_yaxis()
    ax.set(title=f"Weighted score (price weight {price_weight:.2f}) – winner: {parts.sum(axis=1).idxmax()}",
           xlabel="points out of 10")
    ax.legend(frameon=False, ncol=5, loc="lower center", bbox_to_anchor=(0.5, -0.45))
    plt.show()

interact(show_rating, price_weight=FloatSlider(value=0.30, min=0.0, max=0.9, step=0.05,
                                               description="price weight", continuous_update=False));

### 2b · Landed cost: the cheapest per tonne *delivered*

The price at the mill is not what Kaya pays. **Landed cost per tonne = price + freight + product in transit.**

**Product in transit** is the money tied up while the flour is on the truck: **value per tonne × holding rate per year × days in transit ÷ 365.** Kaya's holding rate is 20 % a year.

**✏️ YOUR TURN (two blanks):** first the product in transit, then the landed cost.

In [ ]:
# ── 2b · Landed cost per tonne ──────────────────────────────────────────────────
HOLD = 0.20      # holding rate: 20 % of the value per year

# ✏️ YOUR TURN 1: value per tonne × holding rate × days ÷ 365
mills["transit"] = mills["price"] * ___ * mills["days"] / 365

# ✏️ YOUR TURN 2: landed cost = price + freight + product in transit
mills["landed"] = mills["price"] + mills["freight"] + ___

# Money in two decimals, as on paper (so the next steps use the same numbers as your hand calculation)
mills[["transit", "landed"]] = mills[["transit", "landed"]].round(2)

print(mills[["price", "freight", "transit", "landed"]].round(2))

In [ ]:
# CHECK 2b
try:
    check("product in transit, Großmühle (€/t)", mills.loc["Großmühle", "transit"], 2.68, hint="price × HOLD × days ÷ 365")
    check("landed cost, Bergheim (€/t)", mills.loc["Bergheim", "landed"], 585.92, hint="price + freight + transit")
    check("landed cost, Hofmühle (€/t)", mills.loc["Hofmühle", "landed"], 590.32)
    check("landed cost, Großmühle (€/t)", mills.loc["Großmühle", "landed"], 567.68)
except Exception as e:
    not_yet(e)

In [ ]:
# ── 2b · Picture: what one tonne costs at Kaya's door ───────────────────────────
fig, ax = plt.subplots(figsize=(8, 4))
x = mills.index
ax.bar(x, mills["price"], color=GREY, label="price at the mill")
ax.bar(x, mills["freight"], bottom=mills["price"], color=DARK, label="freight")
ax.bar(x, mills["transit"], bottom=mills["price"] + mills["freight"], color=RED, label="product in transit")
for i, v in enumerate(mills["landed"]):
    ax.text(i, v + 6, f"{v:,.2f} €", ha="center", fontsize=13, fontweight="bold")
ax.set_ylim(400, 640)        # the axis starts at 400 € so the differences are visible – mind this when you read it!
ax.set(title="Landed cost per tonne: the price gap shrinks at the door", ylabel="€ per tonne (axis starts at 400)")
ax.legend(frameon=False, ncol=3, loc="upper center", bbox_to_anchor=(0.5, -0.1)); plt.show()

gap_price = mills.loc["Bergheim", "price"] - mills.loc["Großmühle", "price"]
gap_landed = mills.loc["Bergheim", "landed"] - mills.loc["Großmühle", "landed"]
print(f"Großmühle is {gap_price:.0f} €/t cheaper at the mill, but only {gap_landed:.2f} €/t cheaper at Kaya's door.")

### 2c · Risk: what does a three-week outage cost?

Exhibit D: the chance that one mill stops for three weeks is **10 % a year**. Three weeks = 18 baking nights without flour. Each night Kaya loses 3,300 rolls × 0.22 € margin, and FrischMarkt charges a **1,000 € credit** per missed night (annex Q).

**Expected cost of a disruption = probability per year × cost per event.** Compare it with the yearly price of keeping **Hofmühle as a second mill for 40 %** of the flour (its higher landed cost on that 40 % + 300 € paperwork).

**✏️ YOUR TURN:** the expected cost with one mill only.

In [ ]:
# ── 2c · Expected cost of a disruption ─────────────────────────────────────────
P_OUTAGE = 0.10                                   # chance of a three-week outage per year
OUTAGE_COST = 18 * 3300 * 0.22 + 18 * 1000        # lost margin + FrischMarkt credits, € per outage
print(f"One outage costs {OUTAGE_COST:,.2f} €")

# ✏️ YOUR TURN: probability × cost per event
expected_one_mill = ___ * OUTAGE_COST

# The yearly price of a second mill: 40 % of the flour at Hofmühle's higher landed cost, plus paperwork
second_mill = NEED_T * 0.40 * (mills.loc["Hofmühle", "landed"] - mills.loc["Bergheim", "landed"]) + 300
print(f"Expected outage cost with one mill: {expected_one_mill:,.2f} € a year")
print(f"Price of keeping Hofmühle as second mill: {second_mill:,.2f} € a year")

In [ ]:
# CHECK 2c
try:
    check("expected cost, one mill (€/year)", expected_one_mill, 3106.8, hint="P_OUTAGE × OUTAGE_COST")
    check("second mill (€/year)", second_mill, 369.7)
except Exception as e:
    not_yet(e)

In [ ]:
# ── 2c · Picture: the whole year, three plans ──────────────────────────────────
# If Bergheim stops, Hofmühle can cover 60 % of the need, so only 40 % of an outage still hurts.
plans = pd.DataFrame({
    "flour (landed)": [NEED_T * mills.loc["Großmühle", "landed"],
                       NEED_T * mills.loc["Bergheim", "landed"],
                       NEED_T * 0.6 * mills.loc["Bergheim", "landed"] + NEED_T * 0.4 * mills.loc["Hofmühle", "landed"]],
    "stock of big lots": [10 / 2 * mills.loc["Großmühle", "landed"] * HOLD,     # average stock = half a lot
                          1 / 2 * mills.loc["Bergheim", "landed"] * HOLD,
                          (1 / 2 * mills.loc["Bergheim", "landed"] + 0.5 / 2 * mills.loc["Hofmühle", "landed"]) * HOLD],
    "expected outage": [expected_one_mill, expected_one_mill, P_OUTAGE * 0.4 * OUTAGE_COST],
    "paperwork": [0, 0, 300],
}, index=["All Großmühle", "All Bergheim", "60 % Bergheim\n+ 40 % Hofmühle"])
plans["total"] = plans.sum(axis=1)

fig, ax = plt.subplots(figsize=(9, 4.2))
bottom = np.zeros(3)
for col, shade in zip(plans.columns[:-1], [GREY, "#5f5f5f", RED, "#e07a8c"]):
    ax.bar(plans.index, plans[col], bottom=bottom, color=shade, label=col)
    bottom += plans[col].values
for i, v in enumerate(plans["total"]):
    ax.text(i, v + 150, f"{v:,.0f} €", ha="center", fontsize=13, fontweight="bold")
ax.set_ylim(20000, 27500)    # the axis starts at 20,000 € so the differences are visible
ax.set(title="Cost of flour for a whole year, risk included", ylabel="€ per year (axis starts at 20,000)")
ax.legend(frameon=False, ncol=4, loc="upper center", bbox_to_anchor=(0.5, -0.12)); plt.show()
print(plans.round(0))

### 2d · CO₂: the truck kilometres

**CO₂ of transport = tonne-kilometres × emission factor.** One tonne-kilometre (tkm) = one tonne moved one kilometre. Week 3's truck factor: **0.05789 kg CO₂e per tkm**. (The flour itself has the same footprint at all three mills, so only the truck makes a difference.)

**✏️ YOUR TURN:** multiply the tonne-kilometres by the factor.

In [ ]:
# ── 2d · Truck CO₂ per year for the three plans ─────────────────────────────────
TRUCK = 0.05789                                             # kg CO₂e per tonne-km
tkm = pd.Series({"All Großmühle": NEED_T * 320,
                 "All Bergheim": NEED_T * 150,
                 "60 % Bergheim\n+ 40 % Hofmühle": NEED_T * 0.6 * 150 + NEED_T * 0.4 * 30})

# ✏️ YOUR TURN: tonne-km × factor
co2_kg = tkm * ___

fig, ax = plt.subplots(figsize=(8, 3.6))
bars = ax.bar(co2_kg.index, co2_kg.values, color=[DARK, GREY, GREEN])
ax.bar_label(bars, labels=[f"{v:,.0f} kg" for v in co2_kg.values], fontsize=13, fontweight="bold")
ax.set(title="Truck CO₂ per year (kg CO₂e)", ylabel="kg CO₂e"); plt.show()

In [ ]:
# CHECK 2d
try:
    check("CO₂, all Großmühle (kg)", co2_kg.iloc[0], 733.6, tol=0.1, hint="tkm × TRUCK")
    check("CO₂, 60/40 plan (kg)", co2_kg.iloc[2], 233.8, tol=0.1)
except Exception as e:
    not_yet(e)

## Part 3 · 🧩 Your one-page dashboard for Aylin

Run the cell: it puts your four results on one page – the page a supply manager would bring to Aylin's desk. Then write **three sentences for Aylin** below it: which mill(s), why (name two charts), and what it costs or saves.

In [ ]:
# ── 3 · Four charts, one page ───────────────────────────────────────────────────
fig, ax = plt.subplots(2, 2, figsize=(13, 8))
colours = [MILL_COLOURS[m] for m in mills.index]

b = ax[0, 0].barh(weighted.index, weighted.values, color=colours); ax[0, 0].invert_yaxis()
ax[0, 0].bar_label(b, fmt="%.2f", fontweight="bold"); ax[0, 0].set(title="1 · Factor rating (higher = better)", xlim=(0, 10))

b = ax[0, 1].bar(mills.index, mills["landed"], color=colours); ax[0, 1].set_ylim(500, 610)
ax[0, 1].bar_label(b, fmt="%.2f €", fontweight="bold"); ax[0, 1].set(title="2 · Landed cost per tonne (axis from 500 €)")

plan_colours = [DARK, GREY, GREEN]
b = ax[1, 0].bar(plans.index, plans["total"], color=plan_colours); ax[1, 0].set_ylim(20000, 27500)
ax[1, 0].bar_label(b, labels=[f"{v:,.0f} €" for v in plans["total"]], fontweight="bold")
ax[1, 0].set(title="3 · Year cost incl. expected outage (axis from 20,000 €)")

b = ax[1, 1].bar(co2_kg.index, co2_kg.values, color=plan_colours)
ax[1, 1].bar_label(b, labels=[f"{v:,.0f} kg" for v in co2_kg.values], fontweight="bold")
ax[1, 1].set(title="4 · Truck CO₂ per year")

fig.suptitle("Backhaus Kaya · the mill question in four charts", fontsize=16, fontweight="bold", color=RED)
plt.tight_layout(); plt.show()

**✍️ Three sentences for Aylin** (double-click this cell to write here):

1. Our recommendation:
2. Because (name two charts):
3. It costs / saves per year:

## Part 4 · ⭐ Extra: 1,000 years of flour risk

An **expected** cost is an average. In a real year an outage either happens or it does not. This cell plays 1,000 random years: in each year each mill stops with a 10 % chance. Drag the slider to change the share that goes to Hofmühle (at most 60 %).

In [ ]:
# ── 4 · Monte Carlo: 1,000 random years ────────────────────────────────────────
rng = np.random.default_rng(6)          # fixed seed: everyone sees the same 1,000 years
b_down = rng.random(1000) < P_OUTAGE    # True in the years when Bergheim stops
h_down = rng.random(1000) < P_OUTAGE    # True in the years when Hofmühle stops

def show_years(share_hofmuehle=0.40):
    # If Bergheim stops, Hofmühle can double its share, but never above 60 % of the need.
    cover = min(2 * share_hofmuehle, 0.6)
    lost = np.where(b_down & h_down, 1.0,              # both mills down: the whole outage hurts
           np.where(b_down, 1 - cover, 0.0))           # only Bergheim down: Hofmühle covers part
    # (only Hofmühle down: Bergheim takes over everything, nothing lost)
    outage = lost * OUTAGE_COST
    flour = NEED_T * ((1 - share_hofmuehle) * mills.loc["Bergheim", "landed"] + share_hofmuehle * mills.loc["Hofmühle", "landed"])
    year = flour + outage + (300 if share_hofmuehle > 0 else 0)
    fig, ax = plt.subplots(figsize=(9, 3.8))
    ax.hist(year, bins=30, color=GREY)
    ax.axvline(year.mean(), color=RED, lw=3, label=f"average {year.mean():,.0f} €")
    ax.set(title=f"1,000 years with {share_hofmuehle:.0%} at Hofmühle: most years are calm, a few are very expensive",
           xlabel="flour cost of one year incl. outages (€)", ylabel="number of years")
    ax.legend(frameon=False); plt.show()
    print(f"Worst year: {year.max():,.0f} €   ·   years with an outage loss: {(outage > 0).sum()} of 1,000")

interact(show_years, share_hofmuehle=FloatSlider(value=0.40, min=0.0, max=0.6, step=0.1,
                                                 description="Hofmühle share", continuous_update=False));

---
*Transparency note: Bei der Erstellung dieses Notebooks wurde Claude (Anthropic) zur Programmierung, für Beispiele und als Reviewer (Rechen- und Konsistenzchecks) genutzt. Alle Inhalte wurden von Roman Bartnik geprüft, überarbeitet und verantwortet.*

**Sources:** Heizer, J., Render, B., & Munson, C. (2023). *Operations management: Sustainability and supply chain management* (14th ed., Global ed.). Pearson. · Sterman, J. D. (1989). Modeling managerial behavior: Misperceptions of feedback in a dynamic decision making experiment. *Management Science, 35*(3), 321–339. https://doi.org/10.1287/mnsc.35.3.321 · Lee, H. L., Padmanabhan, V., & Whang, S. (1997). Information distortion in a supply chain: The bullwhip effect. *Management Science, 43*(4), 546–558. https://doi.org/10.1287/mnsc.43.4.546